In [ ]:
# Paramdeep Choudhary
# cu24250162
# labsheet 09

In [ ]:
# Experiment No. 9
# Data Storytelling and Business Insight Generation using Interactive Visualizations

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from google.colab import files
from IPython.display import display

# Upload dataset
uploaded = files.upload()
file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

# Basic information
print("Dataset Shape:", df.shape)
display(df.head())
print("\nMissing Values:")
display(df.isnull().sum())

# Data cleaning
df['date_added'] = pd.to_datetime(df['date_added'], errors='coerce')
df['year_added'] = df['date_added'].dt.year
df['release_year'] = pd.to_numeric(df['release_year'], errors='coerce')
df['rating'] = df['rating'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')

# Remove duplicate records
df = df.drop_duplicates()

# ---------------- KPIs ----------------
total_titles = df['show_id'].nunique()
total_movies = (df['type'] == 'Movie').sum()
total_tv = (df['type'] == 'TV Show').sum()
total_countries = df['country'].nunique()
latest_year = df['release_year'].max()
oldest_year = df['release_year'].min()

print("\nKEY PERFORMANCE INDICATORS")
print("Total Titles:", total_titles)
print("Total Movies:", total_movies)
print("Total TV Shows:", total_tv)
print("Countries Represented:", total_countries)
print("Oldest Release Year:", int(oldest_year))
print("Latest Release Year:", int(latest_year))

# ---------------- 1. Movies vs TV Shows ----------------
type_count = df['type'].value_counts().reset_index()
type_count.columns = ['Type', 'Count']

fig1 = px.pie(
    type_count,
    names='Type',
    values='Count',
    title='Netflix Content Distribution: Movies vs TV Shows',
    hole=0.4
)
fig1.show()

# ---------------- 2. Titles by Release Year ----------------
year_count = df['release_year'].value_counts().sort_index().reset_index()
year_count.columns = ['Release Year', 'Titles']

fig2 = px.line(
    year_count,
    x='Release Year',
    y='Titles',
    markers=True,
    title='Netflix Titles Released Over the Years'
)
fig2.show()

# ---------------- 3. Content Added to Netflix ----------------
added_year = df['year_added'].value_counts().sort_index().reset_index()
added_year.columns = ['Year Added', 'Titles']

fig3 = px.bar(
    added_year,
    x='Year Added',
    y='Titles',
    title='Number of Titles Added to Netflix by Year',
    text='Titles'
)
fig3.show()

# ---------------- 4. Top Countries ----------------
country_df = df[df['country'] != 'Unknown'].copy()
country_df['country_list'] = country_df['country'].str.split(',')

country_expanded = country_df.explode('country_list')
country_expanded['country_list'] = country_expanded['country_list'].str.strip()

top_countries = (
    country_expanded['country_list']
    .value_counts()
    .head(10)
    .reset_index()
)
top_countries.columns = ['Country', 'Titles']

fig4 = px.bar(
    top_countries.sort_values('Titles'),
    x='Titles',
    y='Country',
    orientation='h',
    title='Top 10 Countries by Number of Netflix Titles',
    text='Titles'
)
fig4.show()

# ---------------- 5. Top Genres ----------------
genre_df = df[df['listed_in'].notna()].copy()
genre_df['genre_list'] = genre_df['listed_in'].str.split(',')

genre_expanded = genre_df.explode('genre_list')
genre_expanded['genre_list'] = genre_expanded['genre_list'].str.strip()

top_genres = (
    genre_expanded['genre_list']
    .value_counts()
    .head(10)
    .reset_index()
)
top_genres.columns = ['Genre', 'Titles']

fig5 = px.bar(
    top_genres.sort_values('Titles'),
    x='Titles',
    y='Genre',
    orientation='h',
    title='Top 10 Netflix Genres',
    text='Titles'
)
fig5.show()

# ---------------- 6. Rating Distribution ----------------
rating_count = df['rating'].value_counts().reset_index()
rating_count.columns = ['Rating', 'Titles']

fig6 = px.bar(
    rating_count,
    x='Rating',
    y='Titles',
    title='Netflix Content by Rating',
    text='Titles'
)
fig6.show()

# ---------------- 7. Movies vs TV Shows by Year ----------------
year_type = (
    df.groupby(['release_year', 'type'])
    .size()
    .reset_index(name='Titles')
)

fig7 = px.line(
    year_type,
    x='release_year',
    y='Titles',
    color='type',
    markers=True,
    title='Movies and TV Shows Released by Year'
)
fig7.show()

# ---------------- 8. Country vs Content Type ----------------
top_country_names = top_countries['Country'].head(10).tolist()

country_type = country_expanded[
    country_expanded['country_list'].isin(top_country_names)
]

country_type = (
    country_type.groupby(['country_list', 'type'])
    .size()
    .reset_index(name='Titles')
)

country_type.columns = ['Country', 'Type', 'Titles']

fig8 = px.bar(
    country_type,
    x='Country',
    y='Titles',
    color='Type',
    barmode='group',
    title='Movies vs TV Shows in Top Countries'
)
fig8.show()

# ---------------- 9. Heatmap ----------------
heatmap_data = pd.crosstab(
    df['release_year'],
    df['type']
).tail(25)

fig9 = px.imshow(
    heatmap_data.T,
    aspect='auto',
    title='Heatmap of Recent Content by Type',
    labels={'x': 'Release Year', 'y': 'Content Type', 'color': 'Titles'}
)
fig9.show()

# ---------------- 10. Content Duration ----------------
movie_duration = df[df['type'] == 'Movie'].copy()
movie_duration['duration_num'] = pd.to_numeric(
    movie_duration['duration'].str.extract(r'(\d+)')[0],
    errors='coerce'
)

fig10 = px.histogram(
    movie_duration,
    x='duration_num',
    nbins=30,
    title='Distribution of Movie Durations',
    labels={'duration_num': 'Duration (minutes)'}
)
fig10.show()

# ---------------- Business Insights ----------------
print("\nBUSINESS INSIGHTS")

movie_percentage = total_movies / total_titles * 100
tv_percentage = total_tv / total_titles * 100

print(f"1. Movies represent {movie_percentage:.2f}% of the Netflix catalog, while TV Shows represent {tv_percentage:.2f}%.")

if len(top_countries) > 0:
    print(f"2. {top_countries.iloc[0]['Country']} is the largest contributor among the top countries with approximately {int(top_countries.iloc[0]['Titles'])} titles.")

if len(top_genres) > 0:
    print(f"3. The most common genre/category is {top_genres.iloc[0]['Genre']}.")

print("4. Netflix content production and acquisition increased significantly during recent years, showing expansion of the streaming catalog.")

if not movie_duration['duration_num'].dropna().empty:
    avg_duration = movie_duration['duration_num'].mean()
    print(f"5. The average movie duration is approximately {avg_duration:.0f} minutes.")

print("6. The catalog contains content across multiple countries, indicating a strong international content strategy.")

print("7. Ratings show that Netflix serves different audience segments, from children and families to mature audiences.")

# ---------------- Recommendations ----------------
print("\nACTIONABLE RECOMMENDATIONS")

print("1. Increase investment in popular genres to strengthen audience engagement.")
print("2. Expand local and regional content in countries with growing representation.")
print("3. Maintain a balanced mix of Movies and TV Shows to serve different viewing preferences.")
print("4. Use audience and genre trends to guide future content acquisition and production.")
print("5. Develop targeted recommendations based on content type, genre, country and rating.")
print("6. Monitor changes in content production over time to identify emerging market opportunities.")

# ---------------- Summary Dashboard ----------------
fig_dashboard = go.Figure()

fig_dashboard.add_trace(go.Indicator(
    mode="number",
    value=total_titles,
    title={"text": "Total Titles"},
    domain={'x': [0, 0.25], 'y': [0.5, 1]}
))

fig_dashboard.add_trace(go.Indicator(
    mode="number",
    value=total_movies,
    title={"text": "Movies"},
    domain={'x': [0.25, 0.5], 'y': [0.5, 1]}
))

fig_dashboard.add_trace(go.Indicator(
    mode="number",
    value=total_tv,
    title={"text": "TV Shows"},
    domain={'x': [0.5, 0.75], 'y': [0.5, 1]}
))

fig_dashboard.add_trace(go.Indicator(
    mode="number",
    value=total_countries,
    title={"text": "Countries"},
    domain={'x': [0.75, 1], 'y': [0.5, 1]}
))

fig_dashboard.update_layout(
    title="Netflix Data Storytelling KPI Dashboard",
    height=400
)

fig_dashboard.show()

Saving netflix_titles 2.csv to netflix_titles 2.csv
Dataset Shape: (8807, 12)


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...



Missing Values:


,0
show_id,0
type,0
title,0
director,2634
cast,825
country,831
date_added,10
release_year,0
rating,4
duration,3



KEY PERFORMANCE INDICATORS
Total Titles: 8807
Total Movies: 6131
Total TV Shows: 2676
Countries Represented: 749
Oldest Release Year: 1925
Latest Release Year: 2021



BUSINESS INSIGHTS
1. Movies represent 69.62% of the Netflix catalog, while TV Shows represent 30.38%.
2. United States is the largest contributor among the top countries with approximately 3690 titles.
3. The most common genre/category is International Movies.
4. Netflix content production and acquisition increased significantly during recent years, showing expansion of the streaming catalog.
5. The average movie duration is approximately 100 minutes.
6. The catalog contains content across multiple countries, indicating a strong international content strategy.
7. Ratings show that Netflix serves different audience segments, from children and families to mature audiences.

ACTIONABLE RECOMMENDATIONS
1. Increase investment in popular genres to strengthen audience engagement.
2. Expand local and regional content in countries with growing representation.
3. Maintain a balanced mix of Movies and TV Shows to serve different viewing preferences.
4. Use audience and genre trends to guide futu

In [ ]:
# ============================================================
# QUESTION ANSWERS
# ============================================================

# Q1. What is data storytelling? How is it different from data visualization?
#
# Answer:
# Data storytelling is the process of combining data, visualizations,
# and a meaningful narrative to communicate insights effectively.
# It explains not only what the data shows but also why the findings
# are important and what actions should be taken.
#
# Data visualization mainly represents data through charts, graphs,
# maps, and dashboards. Data storytelling goes one step further by
# adding context, explanation, interpretation, and recommendations.
# Therefore, visualization presents the information, while storytelling
# explains the meaning and business importance of that information.


# Q2. Why is storytelling important in business analytics and decision-making?
#
# Answer:
# Storytelling is important because business stakeholders may not have
# technical knowledge of statistics, programming, or data analysis.
# A well-structured data story converts complex analytical results into
# simple and understandable information.
#
# It helps managers identify business problems, understand trends,
# recognize opportunities and risks, and make better data-driven decisions.
# It also improves communication between data analysts and business teams.
# Therefore, storytelling makes analytical findings more useful and actionable.


# Q3. What are the essential components of an effective data story?
#
# Answer:
# The essential components of an effective data story are:
#
# 1. Data - Accurate and relevant data should be used for analysis.
# 2. Context - The background and business problem should be explained.
# 3. Visualization - Charts and graphs should clearly represent findings.
# 4. Narrative - The insights should be connected through a logical story.
# 5. Key Insights - Important trends, patterns, and problems should be highlighted.
# 6. Recommendations - Practical actions should be suggested based on the findings.
# 7. Conclusion - The main message of the analysis should be summarized clearly.
#
# These components help stakeholders understand the complete journey
# from the business problem to the final recommendation.


# Q4. How do Key Performance Indicators (KPIs) enhance business reporting?
#
# Answer:
# Key Performance Indicators, or KPIs, are measurable values used to
# evaluate the performance of a business, department, or activity.
#
# KPIs make business reporting more focused because they highlight the
# most important metrics. They help managers monitor performance,
# compare current results with previous periods, identify problems,
# and measure progress toward business objectives.
#
# Examples of KPIs include total sales, profit, revenue growth,
# customer satisfaction, number of customers, and conversion rate.
# A KPI dashboard allows decision-makers to understand performance quickly.


# Q5. Why should visualizations be arranged in a logical sequence while
# presenting analytical findings?
#
# Answer:
# Visualizations should be arranged in a logical sequence so that the
# audience can easily follow the data story from beginning to end.
#
# A good sequence usually starts with the business problem, followed
# by an overview of the data, important trends, detailed analysis,
# key findings, and finally recommendations.
#
# Randomly arranged charts can confuse the audience and make it difficult
# to understand the main message. A logical sequence creates a clear
# flow and helps stakeholders connect different findings with each other.


# Q6. What factors should be considered while selecting visualizations
# for a business presentation?
#
# Answer:
# Several factors should be considered while selecting a visualization:
#
# 1. Type of data - Numerical, categorical, time-series, or geographic data.
# 2. Purpose - Whether the goal is comparison, trend analysis, distribution,
#    relationship analysis, or showing composition.
# 3. Audience - The visualization should be understandable to the target users.
# 4. Simplicity - Unnecessary complexity should be avoided.
# 5. Readability - Labels, titles, legends, and scales should be clear.
# 6. Accuracy - The visualization should represent the data correctly.
# 7. Business relevance - The chart should support the business objective.
#
# For example, line charts are useful for trends, bar charts for comparisons,
# pie charts for simple proportions, and heatmaps for patterns across variables.


# Q7. Explain how dashboards and storytelling complement each other
# in Business Intelligence.
#
# Answer:
# Dashboards and storytelling work together to improve Business Intelligence.
# A dashboard provides an interactive and visual summary of important
# business metrics, KPIs, trends, and performance indicators.
#
# Storytelling explains what these numbers mean and why they matter.
# Users can interact with a dashboard to explore the data and then use
# the data story to understand the major findings and possible actions.
#
# For example, a sales dashboard may show a decrease in sales in a particular
# region. Storytelling can explain the possible reasons for the decline and
# suggest actions such as improving marketing or changing product strategy.
# Thus, dashboards provide exploration while storytelling provides context.


# Q8. What challenges may arise while communicating analytical insights
# to non-technical stakeholders?
#
# Answer:
# Several challenges can occur when presenting analytical findings
# to non-technical stakeholders.
#
# Common challenges include:
#
# 1. Technical terminology that is difficult to understand.
# 2. Too many charts or excessive information.
# 3. Complex statistical concepts.
# 4. Lack of business context.
# 5. Difficulty understanding correlations or trends.
# 6. Misinterpretation of charts and numbers.
# 7. Lack of attention to the main business problem.
#
# These challenges can be reduced by using simple language, clear charts,
# meaningful KPIs, relevant examples, and a structured narrative.
# The presentation should focus on business impact rather than technical details.


# Q9. Give two real-world examples where data storytelling has influenced
# business or policy decisions.
#
# Answer:
# Example 1 - Netflix:
# Netflix can analyze viewing patterns, genres, countries, ratings,
# and audience preferences to understand what type of content attracts
# viewers. These insights can support decisions related to content
# production, acquisition, and recommendation systems.
#
# Example 2 - COVID-19 Public Health:
# During the COVID-19 pandemic, governments and health organizations
# used dashboards, charts, maps, and time-series visualizations to
# communicate cases, deaths, hospitalizations, and vaccination trends.
# These visual stories helped communicate the situation to the public
# and supported public-health planning and resource allocation.
#
# These examples demonstrate how analytical information can be converted
# into understandable insights for decision-making.


# Q10. How can effective data storytelling improve strategic planning
# and organizational performance?
#
# Answer:
# Effective data storytelling helps organizations understand their current
# performance and identify important trends, opportunities, and risks.
# It provides decision-makers with evidence that can be used for strategic
# planning instead of relying only on assumptions or intuition.
#
# Data storytelling can help organizations:
#
# 1. Identify areas of strong and weak performance.
# 2. Discover new business opportunities.
# 3. Identify potential risks and problems.
# 4. Allocate resources more effectively.
# 5. Set measurable goals using KPIs.
# 6. Improve communication between departments.
# 7. Monitor progress toward strategic objectives.
# 8. Make faster and more informed decisions.
#
# Therefore, effective data storytelling connects data analysis with
# business strategy and helps organizations improve overall performance.